# CT-RATE Lumora Training

This notebook contains the CT-RATE training flow end to end: download, data checks, model definition, two-phase training, and checkpoint saving. The shared CT-RATE data/download helpers live in `ct_rate_utils.py`; the model training code lives here in the notebook.

In [1]:
from pathlib import Path

DATA_DIR = Path("data/ct_rate")
CHECKPOINT_DIR = Path("checkpoints/ct_rate")
FINAL_MODEL_NAME = "ct_rate_vlm_phase2_fully_trained.pt"

BATCH_SIZE = 2
MAX_TEXT_LENGTH = 128
NUM_WORKERS = 0
PHASE1_EPOCHS = 3
PHASE2_EPOCHS = 3
PHASE1_LR = 1e-4
PHASE2_LR = 2e-5
LIMIT_TRAIN = None
LIMIT_VAL = None
SAVE_EPOCH_CHECKPOINTS = False
SAVE_PHASE1_FINAL = False
SAVE_OPTIMIZER = False
RESUME_CHECKPOINT = None

print("Configuration loaded")
print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR.resolve()}")

Configuration loaded
DATA_DIR: /Users/md.nurealamsiddiquee/Projects/lumora/data/ct_rate
CHECKPOINT_DIR: /Users/md.nurealamsiddiquee/Projects/lumora/checkpoints/ct_rate


## Download

Run this after accepting the gated dataset terms. The defaults below download metadata plus a small subset of volumes for a smoke test. Raise the limits only when you are ready for the storage cost.

In [2]:
from ct_rate_utils import download_ct_rate

download_ct_rate(
    data_dir=DATA_DIR,
    limit_train=20,
    limit_valid=5,
)

Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: (Request ID: Root=1-6a2c7ee0-13433fba69f74d5252f11076;6406606d-22d5-4724-9d71-83ad05082ba7)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/datasets/ibrahimhamamci/CT-RATE/resolve/main/dataset/radiology_text_reports/train_reports.csv.
Make sure your token has the correct permissions.)
Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: (Request ID: Root=1-6a2c7ee0-721b138f1cbe059576acc792;9d911674-cb4d-4fff-9789-01f839e0bb44)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/datasets/ibrahimhamamci/CT-RATE/resolve/main/dataset/radiology_text_reports/validation_repor

Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: (Request ID: Root=1-6a2c7ee5-29033b16489dd1755d49bd9e;b3542225-6386-4db1-9330-b2901d8ee1e6)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/datasets/ibrahimhamamci/CT-RATE/resolve/main/dataset/train_fixed/train_1/train_1_a/train_1_a_1.nii.gz.
Make sure your token has the correct permissions.)
Couldn't access the Hub to check for update but local file already exists. Defaulting to existing file. (error: (Request ID: Root=1-6a2c7ee6-555b8e386587406026b1c712;0463923b-8cd7-4b37-ba90-854612877303)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/datasets/ibrahimhamamci/CT-RATE/resolve/main/dataset/train_fixed/train_1/train_1_a/t

SystemExit: Could not download CT-RATE volume 'valid_1_a_2.nii.gz' from the expected valid paths. The dataset layout may have changed, or this token cannot access gated files.

/Users/md.nurealamsiddiquee/Projects/lumora/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Data Pipeline Smoke Test

This checks that CT-RATE report rows match local NIfTI files and produce the same image/text tensors used by the Lumora model.

In [ ]:
from transformers import AutoTokenizer

from ct_rate_utils import CTRateReportDataset, build_entries, get_device

device = get_device()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_entries, train_csv = build_entries(DATA_DIR, "train")
train_dataset = CTRateReportDataset(train_entries[:8], tokenizer, MAX_TEXT_LENGTH)
sample = train_dataset[0]

print(f"Device: {device}")
print(f"Train CSV: {train_csv}")
print(f"Train samples with local files: {len(train_dataset):,}")
print(f"Image tensor shape: {tuple(sample['image'].shape)}")
print(f"Input IDs shape: {tuple(sample['input_ids'].shape)}")

## Model And Training Helpers

In [ ]:
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel

from ct_rate_utils import CTRateReportDataset, build_entries, require_training_files


class MedicalReportGenerator(nn.Module):
    def __init__(self, vocab_size=50257, embed_dim=768):
        super().__init__()
        self.encoder = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        num_ftrs = self.encoder.classifier.in_features
        self.encoder.classifier = nn.Identity()
        self.projector = nn.Linear(num_ftrs, embed_dim)
        self.decoder = GPT2LMHeadModel.from_pretrained("gpt2")
        self.decoder.resize_token_embeddings(vocab_size)

    def forward(self, images, input_ids, attention_mask):
        visual_features = self.encoder(images)
        visual_embeddings = self.projector(visual_features).unsqueeze(1)
        text_embeddings = self.decoder.transformer.wte(input_ids)
        inputs_embeds = torch.cat((visual_embeddings, text_embeddings), dim=1)

        visual_mask = torch.ones((images.size(0), 1), device=images.device)
        extended_mask = torch.cat((visual_mask, attention_mask), dim=1)
        return self.decoder(inputs_embeds=inputs_embeds, attention_mask=extended_mask).logits


def set_phase(model, phase):
    if phase == "warmup":
        print("Phase 1: encoder frozen, training projector and GPT-2.")
        for p in model.encoder.parameters():
            p.requires_grad = False
        for p in model.projector.parameters():
            p.requires_grad = True
        for p in model.decoder.parameters():
            p.requires_grad = True
    elif phase == "joint":
        print("Phase 2: all layers unfrozen.")
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unknown phase: {phase}")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    return filter(lambda p: p.requires_grad, model.parameters())


def run_epoch(model, loader, optimizer, criterion, device, phase_label, train=True):
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        bar = tqdm(loader, desc=phase_label, leave=True)
        for batch in bar:
            images = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            if train:
                optimizer.zero_grad()

            logits = model(images, input_ids, attention_mask)
            shift_logits = logits[:, 1:-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(len(loader), 1)


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, phase1_done, path):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "phase1_done": phase1_done,
        },
        path,
    )
    print(f"Checkpoint saved: {path}")


def build_loaders(data_dir, tokenizer, device, train_csv=None, valid_csv=None, limit_train=None, limit_val=None):
    train_entries, train_csv_path = build_entries(data_dir, "train", train_csv)
    val_entries, val_csv_path = build_entries(data_dir, "valid", valid_csv)

    if limit_train:
        train_entries = train_entries[:limit_train]
    if limit_val:
        val_entries = val_entries[:limit_val]

    train_dataset = CTRateReportDataset(train_entries, tokenizer, MAX_TEXT_LENGTH)
    val_dataset = CTRateReportDataset(val_entries, tokenizer, MAX_TEXT_LENGTH)

    print(f"Train CSV: {Path(train_csv_path).resolve()}")
    print(f"Valid CSV: {Path(val_csv_path).resolve()}")

    pin = device.type == "cuda"
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
    )
    return train_dataset, val_dataset, train_loader, val_loader

## Train

Set `LIMIT_TRAIN` and `LIMIT_VAL` to `None` in the configuration cell when you are ready to train on all downloaded files.

In [ ]:
require_training_files(DATA_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = get_device()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_dataset, val_dataset, train_loader, val_loader = build_loaders(
    DATA_DIR,
    tokenizer,
    device,
    limit_train=LIMIT_TRAIN,
    limit_val=LIMIT_VAL,
)
print(f"Train samples: {len(train_dataset):,} ({len(train_loader):,} batches)")
print(f"Val samples: {len(val_dataset):,} ({len(val_loader):,} batches)")

sample = next(iter(train_loader))
print(f"Sample image tensor: {tuple(sample['image'].shape)}")
print(f"Sample input ids: {tuple(sample['input_ids'].shape)}")

model = MedicalReportGenerator(vocab_size=len(tokenizer)).to(device)
resume_epoch = 0
phase1_done = False
saved_optimizer = None

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)
    state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state)
    if isinstance(checkpoint, dict):
        resume_epoch = checkpoint.get("epoch", 0)
        phase1_done = checkpoint.get("phase1_done", False)
        saved_optimizer = checkpoint.get("optimizer_state_dict")
    print(f"Resumed checkpoint: {RESUME_CHECKPOINT}")

criterion = nn.CrossEntropyLoss(ignore_index=-100)

if phase1_done:
    print("Phase 1 already complete; skipping warmup.")
else:
    trainable = set_phase(model, "warmup")
    optimizer = torch.optim.AdamW(trainable, lr=PHASE1_LR)
    if saved_optimizer is not None and resume_epoch < PHASE1_EPOCHS:
        optimizer.load_state_dict(saved_optimizer)

    train_loss = math.nan
    val_loss = math.nan
    for epoch in range(resume_epoch + 1, PHASE1_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer, criterion, device, f"Train P1 {epoch}")
        val_loss = run_epoch(model, val_loader, optimizer, criterion, device, f"Val P1 {epoch}", train=False)
        if SAVE_EPOCH_CHECKPOINTS:
            save_checkpoint(
                model,
                optimizer,
                epoch,
                train_loss,
                val_loss,
                phase1_done=False,
                path=CHECKPOINT_DIR / f"ct_rate_vlm_phase1_epoch{epoch}_checkpoint.pt",
            )

    if SAVE_PHASE1_FINAL:
        save_checkpoint(
            model,
            optimizer,
            PHASE1_EPOCHS,
            train_loss,
            val_loss,
            phase1_done=True,
            path=CHECKPOINT_DIR / "ct_rate_vlm_phase1_FINAL.pt",
        )

trainable = set_phase(model, "joint")
optimizer = torch.optim.AdamW(trainable, lr=PHASE2_LR)
avg_train_loss = math.nan
avg_val_loss = math.nan

for epoch in range(1, PHASE2_EPOCHS + 1):
    avg_train_loss = run_epoch(model, train_loader, optimizer, criterion, device, f"Train P2 {epoch}")
    avg_val_loss = run_epoch(model, val_loader, optimizer, criterion, device, f"Val P2 {epoch}", train=False)
    if SAVE_EPOCH_CHECKPOINTS:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            avg_train_loss,
            avg_val_loss,
            phase1_done=True,
            path=CHECKPOINT_DIR / f"ct_rate_vlm_phase2_epoch{epoch}_checkpoint.pt",
        )

final_path = CHECKPOINT_DIR / FINAL_MODEL_NAME
checkpoint = {
    "model_state_dict": model.state_dict(),
    "final_train_loss": avg_train_loss,
    "final_val_loss": avg_val_loss,
    "phase1_done": True,
    "config": {
        "data_dir": str(DATA_DIR),
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "batch_size": BATCH_SIZE,
        "max_text_length": MAX_TEXT_LENGTH,
        "num_workers": NUM_WORKERS,
        "phase1_epochs": PHASE1_EPOCHS,
        "phase1_lr": PHASE1_LR,
        "phase2_epochs": PHASE2_EPOCHS,
        "phase2_lr": PHASE2_LR,
        "limit_train": LIMIT_TRAIN,
        "limit_val": LIMIT_VAL,
    },
}
if SAVE_OPTIMIZER:
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
torch.save(checkpoint, final_path)
print(f"Final model saved: {final_path.resolve()}")